# Econometrics Primer

This notebook is written as a guided chapter, not just a code dump. The goal is to connect econometric formulas to the market-risk workflow a beginner quant actually sees: price histories become returns, returns become risk-factor models, models produce forecasts, and forecasts are checked through diagnostics and backtests.

Keep the matching CSV files in the same folder as this notebook. Every `pd.read_csv(...)` call uses a local filename so students can copy this folder anywhere and run the examples without editing paths.

Required files:

- `econometrics_primer.py`
- `econometrics_price_path.csv`
- `econometrics_factor_returns.csv`
- `econometrics_garch_inputs.csv`
- `econometrics_cointegration_series.csv`

The examples cover returns, volatility, VaR exceptions, OLS factor regression, AR(1), EWMA, GARCH, residual diagnostics, and cointegration-style spread analysis.

## Setup

We start by importing the numerical libraries and the companion Python module. The module contains reusable functions, while the notebook tells the story and shows the intermediate outputs. If `matplotlib` is installed, plots will display inline. If it is not installed, the numerical sections still run.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    plt = None
    HAS_MATPLOTLIB = False

import econometrics_primer as ep

DATA_DIR = Path.cwd()
if HAS_MATPLOTLIB:
    plt.style.use("default")

## 1. Returns And Volatility

Market-risk econometrics usually begins with a price history, but risk models are rarely built directly on prices. Prices are often nonstationary: their level can drift for long periods, and their variance can grow with horizon. Returns are usually more stable and easier to model.

For a price series $S_t$, the arithmetic return is

$$r_t=\frac{S_t-S_{t-1}}{S_{t-1}}$$

and the log return is

$$\ell_t=\log\left(\frac{S_t}{S_{t-1}}\right).$$

Arithmetic returns are intuitive for simple percentage gains and losses. Log returns are convenient because they add exactly across time.

The next cell loads `econometrics_price_path.csv`, computes both return definitions, and appends them to the price table. This is the first habit students should build: never calculate a risk statistic before checking how the raw market data was transformed.

In [ ]:
prices = pd.read_csv("econometrics_price_path.csv")
prices["arithmetic_return"] = ep.arithmetic_returns(prices["price"])
prices["log_return"] = ep.log_returns(prices["price"])
prices

Once returns are available, we estimate the sample mean and sample volatility. The volatility calculation uses the sample denominator $n-1$, which is the usual unbiased estimator for variance under iid assumptions. The normal VaR number shown here is deliberately simple: it is a first benchmark, not a complete market-risk model.

In [ ]:
returns = prices["arithmetic_return"].dropna()
daily_mean = ep.sample_mean(returns)
daily_vol = ep.sample_volatility(returns)
annual_vol = ep.annualize_volatility(daily_vol)
var_99 = ep.normal_var_from_pnl_mean_vol(daily_mean, daily_vol, z_alpha=2.326)

pd.DataFrame(
    {
        "metric": ["daily_mean", "daily_volatility", "annualized_volatility", "normal_99pct_var_loss_return"],
        "value": [daily_mean, daily_vol, annual_vol, var_99],
    }
)

The bar chart makes the sign and size of daily returns visible. This matters pedagogically because volatility is not an abstract formula; it is a summary of the dispersion students can see in the return series.

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    colors = ["#B00000" if r < 0 else "#1A6599" for r in returns]
    ax.bar(prices["date"].iloc[1:], returns * 100, color=colors, alpha=0.88)
    ax.axhline(0, color="#22252A", linewidth=0.8)
    ax.set_title("Arithmetic returns")
    ax.set_ylabel("Return (%)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, axis="y", alpha=0.3)
    plt.show()
else:
    print("Install matplotlib to display this plot.")

## 2. OLS Factor Regression

A factor model explains portfolio return using observable risk-factor moves. In its simplest linear form,

$$r_{p,t}=\alpha+\beta_1 x_{1,t}+\beta_2 x_{2,t}+\epsilon_t.$$

Here the factors are market return and interest-rate change. The estimated coefficients tell us how the portfolio tends to respond to those shocks. OLS chooses the coefficients that minimize squared residuals, giving

$$\hat\beta=(X^\top X)^{-1}X^\top y$$

when $X^\top X$ is invertible.

The next cell estimates a two-factor regression. Notice the units: market return is in decimal return units, while the rate factor is a decimal yield change. Unit discipline is crucial because a rate beta can look numerically large simply because the input is measured in small decimal increments.

In [ ]:
factors = pd.read_csv("econometrics_factor_returns.csv")
fit_two_factor = ep.ols_fit(factors["portfolio_return"], factors[["market_return", "rate_change"]])
fit_one_factor = ep.ols_fit(factors["portfolio_return"], factors["market_return"])

pd.DataFrame(
    {
        "parameter": ["alpha", "market_beta", "rate_beta", "r_squared", "residual_volatility"],
        "value": [fit_two_factor.alpha, fit_two_factor.betas[0], fit_two_factor.betas[1], fit_two_factor.r_squared, fit_two_factor.residual_volatility],
    }
)

A one-factor beta model also gives a useful variance decomposition:

$$\operatorname{Var}(r_p)=\beta^2\operatorname{Var}(r_m)+\operatorname{Var}(\epsilon).$$

This separates systematic market risk from residual risk. A high beta can dominate portfolio volatility even when residual noise is small.

In [ ]:
systematic, residual, total = ep.systematic_residual_variance(
    beta=float(fit_one_factor.betas[0]),
    factor_vol=ep.sample_volatility(factors["market_return"]),
    residual_vol=fit_one_factor.residual_volatility,
)
pd.DataFrame({"component": ["systematic", "residual", "total"], "variance": [systematic, residual, total]})

The scatter plot shows the one-factor relationship visually. The fitted line is not a proof of causality; it is a compact summary of historical co-movement. A serious risk analyst still checks residuals, stability, omitted factors, and stress-period behavior.

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(7, 4))
    x = factors["market_return"].to_numpy()
    y = factors["portfolio_return"].to_numpy()
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = fit_one_factor.alpha + fit_one_factor.betas[0] * x_line
    ax.scatter(x * 100, y * 100, color="#1A6599", label="observations")
    ax.plot(x_line * 100, y_line * 100, color="#B00000", label="OLS fit")
    ax.set_xlabel("Market return (%)")
    ax.set_ylabel("Portfolio return (%)")
    ax.set_title("Portfolio beta regression")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False)
    plt.show()
else:
    print("Install matplotlib to display this plot.")

A two-factor regression can also be viewed as a plane. This 3D plot is useful here because the model has two explanatory variables and one response: market return, rate move, and portfolio return. We will use 3D plots in the batch only when that structure genuinely helps the concept.

In [ ]:
if HAS_MATPLOTLIB:
    fig = plt.figure(figsize=(7.2, 5.0))
    ax = fig.add_subplot(111, projection="3d")

    x = factors["market_return"].to_numpy()
    y = factors["rate_change"].to_numpy()
    z = factors["portfolio_return"].to_numpy()
    x_grid, y_grid = np.meshgrid(
        np.linspace(x.min(), x.max(), 18),
        np.linspace(y.min(), y.max(), 18),
    )
    z_grid = fit_two_factor.alpha + fit_two_factor.betas[0] * x_grid + fit_two_factor.betas[1] * y_grid

    ax.scatter(x * 100, y * 10000, z * 100, color="#B00000", s=28, depthshade=True)
    ax.plot_surface(x_grid * 100, y_grid * 10000, z_grid * 100, cmap="viridis", alpha=0.72)
    ax.set_title("Two-factor regression plane")
    ax.set_xlabel("Market return (%)")
    ax.set_ylabel("Rate move (bps)")
    ax.set_zlabel("Portfolio return (%)")
    ax.view_init(elev=23, azim=-135)
    plt.show()
else:
    print("Install matplotlib to display this 3D plot.")

## 3. AR(1), EWMA, And GARCH

Time-series models describe how risk factors evolve through time. An AR(1) model is

$$X_t=c+\phi X_{t-1}+\epsilon_t.$$

If $|\phi|<1$, the process is mean-reverting and has finite long-run variance. Volatility models focus on the conditional variance rather than the conditional mean. EWMA updates variance as

$$\sigma_{t+1}^2=\lambda\sigma_t^2+(1-\lambda)r_t^2,$$

while GARCH(1,1) uses

$$\sigma_t^2=\omega+\alpha\epsilon_{t-1}^2+\beta\sigma_{t-1}^2.$$

These models are useful because market volatility clusters: large moves tend to be followed by periods of elevated risk.

The next cell estimates an AR(1) model on the return series. For teaching purposes we use a small data set, so the estimate should be treated as an example of mechanics rather than a production-quality forecast.

In [ ]:
garch_data = pd.read_csv("econometrics_garch_inputs.csv")
c, phi, innovation_vol = ep.estimate_ar1(garch_data["return"])
ar1_mean, ar1_vol = ep.ar1_moments(c, phi, innovation_vol) if abs(phi) < 1 else (np.nan, np.nan)

pd.DataFrame(
    {
        "metric": ["c", "phi", "innovation_volatility", "stationary_mean", "stationary_volatility"],
        "value": [c, phi, innovation_vol, ar1_mean, ar1_vol],
    }
)

Now we compute EWMA and GARCH volatility paths. Both methods react to recent squared returns, but GARCH separates the constant long-run variance term, shock response, and persistence term. The stationarity condition $\alpha+\beta<1$ prevents conditional variance from drifting without bound.

In [ ]:
ewma_var = ep.ewma_variance(garch_data["return"], decay=0.94)
garch_var = ep.garch_11_variance(garch_data["return"], omega=0.000002, alpha=0.08, beta=0.90)

vols = pd.DataFrame(
    {
        "date": garch_data["date"],
        "return": garch_data["return"],
        "ewma_volatility": np.sqrt(ewma_var),
        "garch_volatility": np.sqrt(garch_var),
    }
)
vols.tail()

The volatility plot helps students see how conditional volatility changes over time. A risk system that uses yesterday's calm volatility after today's large shock can materially understate tomorrow's risk.

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(vols["date"], vols["ewma_volatility"] * 100, label="EWMA", color="#1A6599")
    ax.plot(vols["date"], vols["garch_volatility"] * 100, label="GARCH(1,1)", color="#2D7A47")
    ax.set_title("Conditional volatility paths")
    ax.set_ylabel("Volatility (%)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False)
    plt.show()
else:
    print("Install matplotlib to display this plot.")

## 4. VaR Exceptions And Cointegration-Style Residuals

Econometric models must be checked after they are fitted. For a correct 99 percent VaR model, the exception probability is 1 percent. Over 250 trading days, the exception count is commonly benchmarked against

$$N\sim\operatorname{Binomial}(250,0.01).$$

This does not prove a model is correct, but it gives a transparent first test of unconditional coverage.

The next cell computes the expected number of exceptions and the probability of seeing eight or more exceptions under the correct model. This is the kind of quick numerical check students should be able to do before reaching for more advanced backtesting machinery.

In [ ]:
pd.DataFrame(
    {
        "metric": ["expected_exceptions", "P_exactly_8", "P_8_or_more"],
        "value": [
            ep.expected_exceptions(250, 0.01),
            ep.binomial_probability(250, 8, 0.01),
            ep.binomial_tail_probability(250, 8, 0.01),
        ],
    }
)

Cointegration addresses a different problem: price levels can be nonstationary individually but have a stable long-run relationship. If $X_t$ and $Y_t$ are each $I(1)$, but

$$Y_t-\theta X_t\sim I(0),$$

then the residual spread may be mean-reverting. This idea is common in relative-value trading, pairs trading, and spread-risk monitoring.

In [ ]:
cointegration = pd.read_csv("econometrics_cointegration_series.csv")
theta, residual = ep.cointegration_residual(cointegration["series_y"], cointegration["series_x"])
resid_c, resid_phi, resid_innov_vol = ep.estimate_ar1(residual)

pd.DataFrame(
    {
        "metric": ["theta", "residual_ar1_c", "residual_ar1_phi", "residual_innovation_volatility"],
        "value": [theta, resid_c, resid_phi, resid_innov_vol],
    }
)

The final plot shows both price-level series and the fitted residual spread. A stationary-looking residual is not by itself a formal unit-root test, but it gives students a visual bridge from regression output to the economic idea of a mean-reverting relative-value spread.

In [ ]:
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(2, 1, figsize=(8, 5.2), sharex=True)
    axes[0].plot(cointegration["date"], cointegration["series_x"], label="Series X", color="#1A6599")
    axes[0].plot(cointegration["date"], cointegration["series_y"], label="Series Y", color="#F0A202")
    axes[0].set_title("Two price-level series")
    axes[0].legend(frameon=False)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(cointegration["date"], residual, label="Regression residual", color="#B00000")
    axes[1].axhline(0, color="#22252A", linewidth=0.8)
    axes[1].set_title("Estimated spread")
    axes[1].legend(frameon=False)
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Install matplotlib to display this plot.")